# 🚀 Final Phase — Notebook 1: Complete Training
## Quantum-Enhanced Deep Learning for Night-Time License Plate Recognition

### What this notebook does:
1. **Resumes** the Quantum model training from Epoch 29 → 100 epochs
2. **Trains** an identical Classical baseline (no quantum layer) for fair comparison
3. **Adds** proper train/val/test split (was missing in Phase-1)
4. **Saves** both checkpoints with full training history

---
### ⚙️ Runtime Setup
- **Runtime**: GPU (T4 or better)
- **Estimated Time**: ~4 hours (Quantum) + ~1 hour (Classical)
- **Required**: Your Drive mounted at `/content/drive/MyDrive/`

## Cell 1 — Install Dependencies

In [ ]:
# Install PennyLane (Quantum framework)
!pip install pennylane pennylane-lightning -q
# Install metrics libraries
!pip install jiwer editdistance -q

import os, zipfile, json, time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
print(f'✅ PennyLane version: {qml.__version__}')

## Cell 2 — Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────
# 🔧 EDIT THESE PATHS TO MATCH YOUR GOOGLE DRIVE SETUP
# ─────────────────────────────────────────────────────────────
PROJECT_PATH     = '/content/drive/MyDrive/Quantum_LPR_Project'
CHECKPOINT_DIR   = os.path.join(PROJECT_PATH, 'checkpoints')
HISTORY_DIR      = os.path.join(PROJECT_PATH, 'history')
ZIP_PATH         = '/content/drive/MyDrive/License Plate/wYe7pBJ7-train.zip'
EXTRACT_PATH     = '/content/lpr_train'
CSV_PATH         = '/content/drive/MyDrive/License Plate/Manifests/2_train_hr_images.csv'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# Training Hyperparameters
# ─────────────────────────────────────────────────────────────
BATCH_SIZE    = 16      # Keep 16 — quantum simulation is memory-heavy
LR_INIT       = 0.001
TOTAL_EPOCHS  = 100
PATIENCE      = 10      # Early stopping patience
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15
SEED          = 42

# Checkpoint file names
QUANTUM_CKPT  = '8qubit_model.pth'         # Existing Phase-1 checkpoint
QUANTUM_BEST  = '8qubit_best.pth'          # Best by val CER
CLASSICAL_CKPT= 'classical_model.pth'      # Classical baseline
CLASSICAL_BEST= 'classical_best.pth'       # Best by val CER

print('✅ Paths configured.')
print(f'   Checkpoints → {CHECKPOINT_DIR}')
print(f'   History     → {HISTORY_DIR}')

## Cell 3 — Extract Dataset (Smart: skips if already done)

In [ ]:
def extract_dataset():
    target_check = os.path.join(EXTRACT_PATH, 'train')
    if not os.path.exists(target_check):
        print(f'📂 Extracting dataset ...')
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall(EXTRACT_PATH)
            print(f'✅ Extraction complete.')
        except FileNotFoundError:
            print(f'❌ Zip not found: {ZIP_PATH}')
            return False
    else:
        print(f'✅ Dataset already extracted — skipping.')
    return True

if not extract_dataset():
    raise SystemExit('Stopping — missing dataset.')

## Cell 4 — Dataset Class (with Night Augmentation)

In [ ]:
CHARS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}   # 0 = CTC blank
IDX2CHAR = {i+1: c for i, c in enumerate(CHARS)}

class LPRDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, mode='train'):
        self.data_frame   = pd.read_csv(csv_file)
        self.root_dir     = root_dir
        self.transform    = transform
        self.mode         = mode

    def __len__(self):
        return len(self.data_frame)

    def encode_text(self, text):
        return [CHAR2IDX[c] for c in str(text).upper() if c in CHAR2IDX]

    def make_night(self, img_tensor):
        """Simulate dark / noisy night conditions."""
        if torch.rand(1) < 0.5:
            gamma = np.random.uniform(2.0, 3.5)
            img_tensor = torch.pow(img_tensor, gamma)
            noise = torch.randn_like(img_tensor) * 0.05
            img_tensor = torch.clamp(img_tensor + noise, 0, 1)
        return img_tensor

    def __getitem__(self, idx):
        img_path   = self.data_frame.iloc[idx]['path']
        label_text = self.data_frame.iloc[idx]['label']

        full_path = img_path if img_path.startswith('/content') \
                    else os.path.join(self.root_dir, img_path)
        try:
            image = Image.open(full_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (256, 64))

        if self.transform:
            image = self.transform(image)

        if self.mode == 'train':          # Augment only during training
            image = self.make_night(image)

        encoded = torch.tensor(self.encode_text(label_text), dtype=torch.long)
        length  = torch.tensor(len(encoded), dtype=torch.long)
        return image, encoded, length


def collate_fn(batch):
    images, labels, lengths = zip(*batch)
    images  = torch.stack(images, 0)
    lengths = torch.stack(lengths, 0)
    targets = torch.cat(labels)
    return images, targets, lengths


transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
])

# ── Build full dataset, then split ──────────────────────────
torch.manual_seed(SEED)
full_dataset = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='train')
N = len(full_dataset)
n_train = int(N * TRAIN_RATIO)
n_val   = int(N * VAL_RATIO)
n_test  = N - n_train - n_val

train_set, val_set, test_set = random_split(full_dataset, [n_train, n_val, n_test])

# Val/test — no night augmentation
val_set.dataset  = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='eval')
test_set.dataset = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform, mode='eval')

train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

# Save test indices for reproducibility
test_indices = list(test_set.indices)
with open(os.path.join(PROJECT_PATH, 'test_indices.json'), 'w') as f:
    json.dump(test_indices, f)

print(f'✅ Dataset split: Train={n_train}  Val={n_val}  Test={n_test}')

## Cell 5 — Model Definitions
### A) Shared Components (Enhancer + CNN)

In [ ]:
# ──────────────────────────────────────────────────────────────
# SHARED: Zero-DCE Low-Light Enhancer (identical in both models)
# ──────────────────────────────────────────────────────────────
class ZeroDCE_Light(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(16, 24, 3, 1, 1), nn.Tanh()
        )
    def forward(self, x):
        A = self.conv(x)
        enhanced = x
        for i in range(8):
            r = A[:, 3*i:3*(i+1), :, :]
            enhanced = enhanced + r * (torch.pow(enhanced, 2) - enhanced)
        return enhanced


# ──────────────────────────────────────────────────────────────
# B) QUANTUM MODEL  (Phase-1 architecture, 8 qubits)
# ──────────────────────────────────────────────────────────────
N_QUBITS = 8
N_LAYERS = 2
dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch')
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {'weights': (N_LAYERS, N_QUBITS, 3)}
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
    def forward(self, x):
        return self.q_layer(x)

class HybridLPRNet_8Q(nn.Module):
    """Quantum-Enhanced LPR model (Phase-1 architecture)."""
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 1, 1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128, N_QUBITS, 1, 1)
        )
        self.quantum   = QuantumLayer()
        self.rnn       = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier= nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b, c, h, w = x.size()
        x = x.mean(dim=2).permute(0, 2, 1)    # [B, W, 8]
        x_flat = x.reshape(-1, N_QUBITS)
        q_out  = self.quantum(x_flat)
        q_out  = q_out.reshape(b, w, N_QUBITS)
        rnn_out, _ = self.rnn(q_out)
        out = self.classifier(rnn_out)
        return out.permute(1, 0, 2)            # [W, B, classes]

    def get_quantum_features(self, x):
        """Helper: returns qubit inputs before the quantum layer (for visualization)."""
        with torch.no_grad():
            x = self.enhancer(x)
            x = self.features(x)
            b, c, h, w = x.size()
            return x.mean(dim=2).permute(0, 2, 1)  # [B, W, 8]


# ──────────────────────────────────────────────────────────────
# C) CLASSICAL BASELINE (identical, quantum → classical FC)
# ──────────────────────────────────────────────────────────────
class ClassicalLPRNet(nn.Module):
    """
    Classical baseline for fair comparison.
    IDENTICAL to HybridLPRNet_8Q except the quantum circuit
    is replaced with: Linear(8,16) -> Tanh() -> Linear(16,8)
    This preserves the same input/output shape so the LSTM is unchanged.
    """
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 1, 1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128, N_QUBITS, 1, 1)
        )
        # Classical replacement: same I/O shape as quantum layer
        self.classical = nn.Sequential(
            nn.Linear(N_QUBITS, 16),
            nn.Tanh(),
            nn.Linear(16, N_QUBITS)
        )
        self.rnn       = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier= nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b, c, h, w = x.size()
        x = x.mean(dim=2).permute(0, 2, 1)    # [B, W, 8]
        x = self.classical(x)                  # [B, W, 8] — classical transform
        rnn_out, _ = self.rnn(x)
        out = self.classifier(rnn_out)
        return out.permute(1, 0, 2)            # [W, B, classes]


# Quick parameter count
q_model = HybridLPRNet_8Q(num_classes=37).to(device)
c_model = ClassicalLPRNet(num_classes=37).to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'⚡ Quantum  model params: {count_params(q_model):,}')
print(f'🔷 Classical model params: {count_params(c_model):,}')

## Cell 6 — Utility Functions (Checkpoint + CTC Decode + CER)

In [ ]:
import editdistance

# ── Checkpoint helpers ──────────────────────────────────────
def save_checkpoint(model, optimizer, epoch, val_cer, filename):
    path = os.path.join(CHECKPOINT_DIR, filename)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_cer': val_cer,
    }, path)

def load_checkpoint(model, optimizer, filename):
    path = os.path.join(CHECKPOINT_DIR, filename)
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start = ckpt.get('epoch', -1) + 1
        print(f'  ✅ Loaded checkpoint — resuming from Epoch {start}')
        return start
    print(f'  ⚠️  No checkpoint at {path} — starting fresh.')
    return 0


# ── CTC Greedy Decode ──────────────────────────────────────
def ctc_greedy_decode(log_probs):
    """log_probs: [T, B, C] — returns list of decoded strings"""
    indices = torch.argmax(log_probs, dim=2)   # [T, B]
    decoded_batch = []
    for b in range(indices.size(1)):
        seq = indices[:, b].tolist()
        chars, prev = [], -1
        for idx in seq:
            if idx != 0 and idx != prev:
                chars.append(IDX2CHAR.get(idx, ''))
            prev = idx
        decoded_batch.append(''.join(chars))
    return decoded_batch


def decode_targets(targets, lengths):
    """Decode flat target tensor back to list of strings."""
    strings = []
    offset = 0
    for l in lengths.tolist():
        seq = targets[offset:offset+l].tolist()
        strings.append(''.join(IDX2CHAR.get(i, '') for i in seq))
        offset += l
    return strings


def compute_cer_wer(model, loader):
    """Character Error Rate and Word Error Rate on a dataloader."""
    model.eval()
    total_cer, total_wer, count = 0.0, 0, 0
    with torch.no_grad():
        for imgs, targets, lengths in loader:
            imgs = imgs.to(device)
            preds = model(imgs).cpu()         # [T, B, C]
            decoded = ctc_greedy_decode(preds)
            actuals = decode_targets(targets, lengths)
            for pred, true in zip(decoded, actuals):
                if len(true) > 0:
                    total_cer += editdistance.eval(pred, true) / len(true)
                total_wer += (0 if pred == true else 1)
                count += 1
    return (total_cer / count if count else 1.0,
            total_wer / count if count else 1.0)

print('✅ Utilities loaded.')

## Cell 7 — Generic Training Loop (used by both models)

In [ ]:
def train_model(model, model_name, ckpt_file, best_ckpt_file,
                resume=True):
    """
    Train model for TOTAL_EPOCHS with:
    - Cosine LR annealing
    - Early stopping on val CER
    - Checkpoint every epoch (resume-friendly)
    - Best-model checkpoint by lowest val CER

    Returns: history dict with train_loss, val_loss, val_cer, val_wer per epoch
    """
    optimizer  = optim.Adam(model.parameters(), lr=LR_INIT)
    criterion  = nn.CTCLoss(blank=0, zero_infinity=True)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS)

    start_epoch = 0
    if resume:
        start_epoch = load_checkpoint(model, optimizer, ckpt_file)
        for _ in range(start_epoch):          # Fast-forward LR scheduler
            scheduler.step()

    # Load existing history if resuming
    history_path = os.path.join(HISTORY_DIR, f'{model_name}_history.json')
    if resume and os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        best_cer = min(history['val_cer']) if history['val_cer'] else 1.0
        print(f'  📈 Loaded history — best val CER so far: {best_cer:.4f}')
    else:
        history  = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'val_wer': []}
        best_cer = 1.0

    no_improve = 0

    print(f'\n🚀 Training [{model_name}] — Epochs {start_epoch+1}–{TOTAL_EPOCHS}')
    print('=' * 70)

    for epoch in range(start_epoch, TOTAL_EPOCHS):
        # ── Training ──────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Ep {epoch+1:3d}/{TOTAL_EPOCHS} [{model_name}]')
        for imgs, targets, lengths in pbar:
            imgs    = imgs.to(device)
            targets = targets.to(device)
            lengths = lengths.to(device)

            preds = model(imgs)   # [T, B, C]
            T     = preds.size(0)
            B     = imgs.size(0)
            input_lengths = torch.full((B,), T, dtype=torch.long, device=device)

            loss  = criterion(preds.log_softmax(2), targets, input_lengths, lengths)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}',
                              'lr': f'{scheduler.get_last_lr()[0]:.6f}'})

        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)

        # ── Validation ────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets, lengths in val_loader:
                imgs    = imgs.to(device)
                targets = targets.to(device)
                lengths = lengths.to(device)
                preds   = model(imgs)
                T, B    = preds.size(0), imgs.size(0)
                il      = torch.full((B,), T, dtype=torch.long, device=device)
                val_loss += criterion(preds.log_softmax(2), targets, il, lengths).item()
        avg_val_loss = val_loss / len(val_loader)

        val_cer, val_wer = compute_cer_wer(model, val_loader)

        # ── Record history ───────────────────────────────
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_cer'].append(val_cer)
        history['val_wer'].append(val_wer)

        with open(history_path, 'w') as f:
            json.dump(history, f)

        print(f'  Ep {epoch+1:3d} | TrainLoss={avg_train_loss:.4f} | '
              f'ValLoss={avg_val_loss:.4f} | CER={val_cer:.4f} | WER={val_wer:.4f}')

        # ── Save checkpoint every epoch ──────────────────
        save_checkpoint(model, optimizer, epoch, val_cer, ckpt_file)

        # ── Best model tracking ───────────────────────────
        if val_cer < best_cer:
            best_cer   = val_cer
            no_improve = 0
            save_checkpoint(model, optimizer, epoch, val_cer, best_ckpt_file)
            print(f'  🏆 New best CER={best_cer:.4f} — saved {best_ckpt_file}')
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  ⏹  Early stopping after {PATIENCE} epochs without improvement.')
                break

    print(f'\n✅ [{model_name}] Training complete. Best val CER: {best_cer:.4f}')
    return history

print('✅ Training loop defined.')

## Cell 8 — Train Quantum Model (Resume from Epoch 29)

In [ ]:
print('⚡ Starting QUANTUM model training ...')
quantum_history = train_model(
    model         = q_model,
    model_name    = 'Quantum',
    ckpt_file     = QUANTUM_CKPT,
    best_ckpt_file= QUANTUM_BEST,
    resume        = True    # ← Loads 8qubit_model.pth from Epoch 29
)

## Cell 9 — Train Classical Baseline (Fresh, 100 epochs)

In [ ]:
print('🔷 Starting CLASSICAL baseline training ...')
classical_history = train_model(
    model         = c_model,
    model_name    = 'Classical',
    ckpt_file     = CLASSICAL_CKPT,
    best_ckpt_file= CLASSICAL_BEST,
    resume        = False   # ← Train from scratch
)

## Cell 10 — Training Curves Comparison Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Quantum vs Classical — Training History', fontsize=15, fontweight='bold')

colors = {'Quantum': '#7B2D8B', 'Classical': '#2196F3'}

q_hist = quantum_history
c_hist = classical_history

def safe_plot(ax, q_data, c_data, ylabel, title):
    q_epochs = range(1, len(q_data) + 1)
    c_epochs = range(1, len(c_data) + 1)
    ax.plot(q_epochs, q_data, color=colors['Quantum'],   label='Quantum',   linewidth=2)
    ax.plot(c_epochs, c_data, color=colors['Classical'], label='Classical', linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

safe_plot(axes[0], q_hist['val_loss'], c_hist['val_loss'],
          'CTC Loss',  'Validation Loss')
safe_plot(axes[1], q_hist['val_cer'],  c_hist['val_cer'],
          'CER ↓',     'Character Error Rate (lower = better)')
safe_plot(axes[2], q_hist['val_wer'],  c_hist['val_wer'],
          'WER ↓',     'Word Error Rate (lower = better)')

plt.tight_layout()
fig_path = os.path.join(PROJECT_PATH, 'training_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved training curves → {fig_path}')

## Cell 11 — Quick Sanity Check: Best Epoch Summary

In [ ]:
def best_epoch_summary(history, name):
    best_idx = int(np.argmin(history['val_cer']))
    print(f'\n{'─'*50}')
    print(f'  [{name}] Best Results (Epoch {best_idx+1})')
    print(f'{'─'*50}')
    print(f'  Train Loss : {history["train_loss"][best_idx]:.4f}')
    print(f'  Val Loss   : {history["val_loss"][best_idx]:.4f}')
    print(f'  Val CER    : {history["val_cer"][best_idx]:.4f}  ({history["val_cer"][best_idx]*100:.1f}%)')
    print(f'  Val WER    : {history["val_wer"][best_idx]:.4f}  ({history["val_wer"][best_idx]*100:.1f}%)')
    print(f'  Accuracy   : {(1-history["val_wer"][best_idx])*100:.1f}%  (exact plate match)')

best_epoch_summary(quantum_history,   'Quantum  ⚡')
best_epoch_summary(classical_history, 'Classical 🔷')

print('\n✅ Notebook 1 complete! Run Notebook 2 for full evaluation.')